In [12]:
import xml.etree.ElementTree as ET
import pandas as pd
import numpy as np
import json
import ast
from pathlib import Path

def parse_xml_to_df(xml_path, entity_tag='book'):
    """Parses XML files into a DataFrame handling scalar and list elements."""
    tree = ET.parse(xml_path)
    root = tree.getroot()
    records = []
    
    for entity in root.findall(f'.//{entity_tag}'):
        data = {}
        for child in entity:
            if len(child) > 0:
                vals = [sub.text for sub in child if sub.text]
                data[child.tag] = vals if vals else None
            else:
                data[child.tag] = child.text if child.text else None
        records.append(data)
    return pd.DataFrame(records)

def load_csv_with_json_lists(csv_path):
    """Loads a CSV and safely parses stringified lists or JSON arrays without crashing on plain text."""
    df = pd.read_csv(csv_path)
    
    def parse_list_string(val):
        if isinstance(val, str) and val.strip().startswith('['):
            try:
                # Versuch 1: Standard-JSON (z. B. ["Author 1", "Author 2"])
                return json.loads(val)
            except (json.JSONDecodeError, TypeError):
                try:
                    # Versuch 2: Python-Listen-Syntax (z. B. ['Author 1', 'Author 2'])
                    res = ast.literal_eval(val)
                    if isinstance(res, list):
                        return res
                except (ValueError, SyntaxError):
                    pass
        return val

    for col in df.columns:
        # Prüfen, ob die Spalte mindestens einen Eintrag hat, der wie eine Liste aussieht
        sample = df[col].dropna().iloc[0] if len(df[col].dropna()) > 0 else None
        if isinstance(sample, str) and sample.strip().startswith('['):
            df[col] = df[col].apply(parse_list_string)
            
    return df

def clean_missing_values(df):
    """Replaces empty strings, empty lists, and 'None' strings with np.nan safely."""
    def is_empty(x):
        if x is None:
            return True
        if isinstance(x, (list, np.ndarray, tuple)):
            return len(x) == 0
        if pd.isna(x):
            return True
        if isinstance(x, str):
            return x.strip() in ["", "[]", "None", "nan", "NaN"]
        return False

    return df.map(lambda x: np.nan if is_empty(x) else x)

# ==========================================
# LOAD DATASETS
# ==========================================
dataset_paths = {
    'DBpedia': ('data/dbpedia_book_data.xml', 'xml'),
    'Goodreads': ('data/books_1.Best_Books_Ever.csv', 'csv'),
    'Amazon': ('data/amazon_books_subset.csv', 'csv')
}

datasets = {}

# DBpedia
if Path(dataset_paths['DBpedia'][0]).exists():
    datasets['DBpedia'] = parse_xml_to_df(dataset_paths['DBpedia'][0])

# Goodreads
if Path(dataset_paths['Goodreads'][0]).exists():
    datasets['Goodreads'] = load_csv_with_json_lists(dataset_paths['Goodreads'][0])

# Amazon
if Path(dataset_paths['Amazon'][0]).exists():
    datasets['Amazon'] = load_csv_with_json_lists(dataset_paths['Amazon'][0])

In [13]:
# ==========================================
# MISSING VALUES ANALYSIS (>30% THRESHOLD)
# ==========================================
missing_reports = []

for name, df in datasets.items():
    df_cleaned = clean_missing_values(df)
    missing_pct = (df_cleaned.isnull().mean() * 100).round(2)
    missing_counts = df_cleaned.isnull().sum()
    total_records = len(df_cleaned)
    
    for col in df_cleaned.columns:
        pct = missing_pct[col]
        missing_reports.append({
            'Dataset': name,
            'Attribute Name': col,
            'Missing Count': missing_counts[col],
            'Total Records': total_records,
            'Missing (%)': pct,
            'High Missing (>30%)': pct > 30.0
        })

df_missing_summary = pd.DataFrame(missing_reports)

# Display attributes with >30% missing values
high_missing_df = df_missing_summary[df_missing_summary['High Missing (>30%)']].sort_values(by=['Dataset', 'Missing (%)'], ascending=[True, False])

print("=== ATTRIBUTES WITH > 30% MISSING VALUES ===")
if not high_missing_df.empty:
    display(high_missing_df[['Dataset', 'Attribute Name', 'Missing Count', 'Total Records', 'Missing (%)']])
else:
    print("No attributes found with more than 30% missing values.")

=== ATTRIBUTES WITH > 30% MISSING VALUES ===


,Dataset,Attribute Name,Missing Count,Total Records,Missing (%)
2,DBpedia,year,18567,20981,88.49
4,DBpedia,language,18378,20981,87.59
7,DBpedia,genres,10303,20981,49.11
18,Goodreads,edition,47523,52478,90.56
23,Goodreads,awards,41864,52478,79.77
27,Goodreads,setting,40900,52478,77.94
16,Goodreads,characters,38712,52478,73.77
9,Goodreads,series,29009,52478,55.28
22,Goodreads,firstPublishDate,21326,52478,40.64


In [14]:
# ==========================================
# TABLE 1: DATASET PROFILE & ATTRIBUTE LIST WITH (MV)
# ==========================================
table_1_rows = []

# Identifikatoren, die nicht als Daten-Attribute in Table 1 gezählt werden sollen
IGNORE_ATTRIBUTES = {'amazon_id', 'asin', 'isbn10', 'isbn13'}

for name, df in datasets.items():
    df_cleaned = clean_missing_values(df)
    missing_pct = df_cleaned.isnull().mean() * 100
    
    # Filtere reine Identifikatoren für die Attributzählung heraus
    attr_cols = [col for col in df_cleaned.columns if col not in IGNORE_ATTRIBUTES]
    
    # Format attribute list and mark >30% missing values with (MV)
    formatted_attrs = []
    for col in attr_cols:
        if missing_pct[col] > 30:
            formatted_attrs.append(f"{col} (MV)")
        else:
            formatted_attrs.append(col)
            
    filepath, fileformat = dataset_paths.get(name, ('unknown', 'csv'))
    
    table_1_rows.append({
        'Dataset': name,
        'Source(*)': filepath,
        'Format': fileformat,
        '# of entities': len(df_cleaned),
        '# of attributes': len(attr_cols),
        'List of attributes (**)': ", ".join(formatted_attrs)
    })

df_table_1 = pd.DataFrame(table_1_rows)
print("=== TABLE 1: SCHEMA AND BASIC PROFILE OF EACH DATASET ===")
display(df_table_1)

print("\n=== ATTRIBUTE LISTS PER DATASET ===")
for _, row in df_table_1.iterrows():
    print(f"\n[{row['Dataset']}] ({row['# of attributes']} attributes):")
    print(f"  {row['List of attributes (**)']}")



=== TABLE 1: SCHEMA AND BASIC PROFILE OF EACH DATASET ===


,Dataset,Source(*),Format,# of entities,# of attributes,List of attributes (**)
0,DBpedia,data/dbpedia_book_data.xml,xml,20981,8,"title, publisher, year (MV), pages, language (..."
1,Goodreads,data/books_1.Best_Books_Ever.csv,csv,52478,23,"title, series (MV), author, rating, descriptio..."
2,Amazon,data/amazon_books_subset.csv,csv,40000,11,"title, subtitle, authors, publisher, publicati..."



=== ATTRIBUTE LISTS PER DATASET ===

[DBpedia] (8 attributes):
  title, publisher, year (MV), pages, language (MV), isbns, authors, genres (MV)

[Goodreads] (23 attributes):
  title, series (MV), author, rating, description, language, isbn, genres, characters (MV), bookFormat, edition (MV), pages, publisher, publishDate, firstPublishDate (MV), awards (MV), numRatings, ratingsByStars, likedPercent, setting (MV), bbeScore, bbeVotes, price

[Amazon] (11 attributes):
  title, subtitle, authors, publisher, publication_date, language, pages, genres, average_rating, ratings_count, price


In [15]:
# ==========================================
# 1. DYNAMIC ATTRIBUTE TYPE INFERENCE
# ==========================================
def infer_attribute_type(series):
    """
    Infers semantic attribute type (string, string/list, numeric, date) 
    based on the actual data content of a series.
    """
    valid_vals = series.dropna()
    if len(valid_vals) == 0:
        return 'string'
    
    sample = valid_vals.iloc[0]
    
    # Check for List types
    if isinstance(sample, (list, tuple, np.ndarray)):
        return 'string/list'
    if isinstance(sample, str) and sample.strip().startswith('['):
        return 'string/list'
    
    # Check for Numeric types
    if pd.api.types.is_numeric_dtype(series):
        return 'numeric'
    
    # Check for Date / Year types
    col_name = series.name.lower()
    if 'date' in col_name or 'year' in col_name:
        return 'date'
        
    return 'string'

# ==========================================
# 2. DYNAMIC TABLE 2 GENERATION
# ==========================================
all_columns = set()
for df in datasets.values():
    all_columns.update(df.columns)

table_2_rows = []

for col in sorted(list(all_columns)):
    # Identify which datasets contain this attribute
    found_in = [ds_name for ds_name, df in datasets.items() if col in df.columns]
    
    # Get a sample dataframe that contains this column to infer its type
    sample_df = next(df for df in datasets.values() if col in df.columns)
    attr_type = infer_attribute_type(sample_df[col])
    
    table_2_rows.append({
        'Attribute name': col,
        'Attribute type': attr_type,
        'Datasets in which found': ", ".join(found_in),
        'Count Datasets': len(found_in)
    })

df_table_2 = pd.DataFrame(table_2_rows)

# Display Full Table 2
print("=== TABLE 2: INTEGRATED SCHEMA OVERLAP (ALL ATTRIBUTES) ===")
display(df_table_2[['Attribute name', 'Attribute type', 'Datasets in which found']])

# ==========================================
# 3. OVERLAPPING ATTRIBUTES ONLY (COMPACT TABLE)
# ==========================================
# Filter attributes that appear in 2 or more datasets
df_overlapping = df_table_2[df_table_2['Count Datasets'] >= 2].drop(columns=['Count Datasets'])

print("\n=== OVERLAPPING ATTRIBUTES ONLY (FOUND IN AT LEAST 2 DATASETS) ===")
if not df_overlapping.empty:
    display(df_overlapping)
else:
    print("No exact overlapping attribute names found between the datasets.")

# ==========================================
# 4. ENTITY OVERLAP CHECK (DYNAMIC PAIRWISE MATCH)
# ==========================================
clean_str = lambda s: str(s).lower().strip() if pd.notna(s) else ""

def get_first_author(val):
    if isinstance(val, (list, tuple, np.ndarray)) and len(val) > 0:
        return clean_str(val[0])
    return clean_str(val)

def extract_author_col(df):
    """Finds author/authors/author_name column dynamically."""
    for col in ['authors', 'author', 'author_name']:
        if col in df.columns:
            return df[col]
    return pd.Series([""] * len(df))

print("\n=== ENTITY OVERLAP CHECK ===")
ds_names = list(datasets.keys())

for i in range(len(ds_names)):
    for j in range(i + 1, len(ds_names)):
        name1, name2 = ds_names[i], ds_names[j]
        df1, df2 = datasets[name1], datasets[name2]
        
        # Title Exact Match
        titles1 = set(df1['title'].apply(clean_str)) - {""}
        titles2 = set(df2['title'].apply(clean_str)) - {""}
        overlap_titles = titles1.intersection(titles2)
        
        # Title + First Author Match
        key1 = df1['title'].apply(clean_str) + " | " + extract_author_col(df1).apply(get_first_author)
        key2 = df2['title'].apply(clean_str) + " | " + extract_author_col(df2).apply(get_first_author)
        
        keys1 = set(key1[df1['title'].notnull()]) - {" | "}
        keys2 = set(key2[df2['title'].notnull()]) - {" | "}
        overlap_keys = keys1.intersection(keys2)
        
        print(f"Overlap {name1} <-> {name2}:")
        print(f"  - By Title (Exact):           {len(overlap_titles)} books")
        print(f"  - By Title + First Author:    {len(overlap_keys)} books")

=== TABLE 2: INTEGRATED SCHEMA OVERLAP (ALL ATTRIBUTES) ===


,Attribute name,Attribute type,Datasets in which found
0,amazon_id,numeric,Amazon
1,asin,string,Amazon
2,author,string,Goodreads
3,authors,string/list,"DBpedia, Amazon"
4,average_rating,numeric,Amazon
5,awards,string/list,Goodreads
6,bbeScore,numeric,Goodreads
7,bbeVotes,numeric,Goodreads
8,bookFormat,string,Goodreads
9,characters,string/list,Goodreads



=== OVERLAPPING ATTRIBUTES ONLY (FOUND IN AT LEAST 2 DATASETS) ===


,Attribute name,Attribute type,Datasets in which found
3,authors,string/list,"DBpedia, Amazon"
13,genres,string/list,"DBpedia, Goodreads, Amazon"
18,language,string,"DBpedia, Goodreads, Amazon"
21,pages,string,"DBpedia, Goodreads, Amazon"
22,price,string,"Goodreads, Amazon"
25,publisher,string,"DBpedia, Goodreads, Amazon"
32,title,string,"DBpedia, Goodreads, Amazon"



=== ENTITY OVERLAP CHECK ===
Overlap DBpedia <-> Goodreads:
  - By Title (Exact):           2857 books
  - By Title + First Author:    1208 books
Overlap DBpedia <-> Amazon:
  - By Title (Exact):           653 books
  - By Title + First Author:    502 books
Overlap Goodreads <-> Amazon:
  - By Title (Exact):           3401 books
  - By Title + First Author:    1007 books


In [1]:
import pandas as pd, numpy as np

gr = pd.read_csv("data/books_1.Best_Books_Ever.csv")
am = pd.read_csv("data/amazon_books_subset.csv")

print(len(gr), len(gr.columns))   # erwartet: 52478, 23
print(len(am), len(am.columns))
print(am.columns.tolist())        # welche isbn/id-Spalten gibt es genau?

52478 23
40000 15
['amazon_id', 'asin', 'title', 'subtitle', 'authors', 'publisher', 'publication_date', 'language', 'pages', 'genres', 'average_rating', 'ratings_count', 'price', 'isbn13', 'isbn10']


In [2]:
def mv(df):
    d = df.replace(r"^\s*$", np.nan, regex=True).replace({"[]": np.nan})
    return (d.isna().mean() * 100).round(1).sort_values(ascending=False)

print(mv(am))
print(mv(gr))   # zur Kontrolle gegen die MV-Markierungen in Table 1

isbn10              23.5
pages               23.4
price               19.9
isbn13              19.4
subtitle            15.4
publisher           14.7
language            14.6
publication_date    14.5
genres               8.8
amazon_id            0.0
asin                 0.0
title                0.0
authors              0.0
average_rating       0.0
ratings_count        0.0
dtype: float64
edition             90.6
awards              79.8
setting             77.9
characters          73.8
series              55.3
firstPublishDate    40.6
price               27.4
genres               8.8
language             7.3
publisher            7.0
pages                4.5
bookFormat           2.8
ratingsByStars       2.7
description          2.5
publishDate          1.7
likedPercent         1.2
isbn                 0.0
rating               0.0
numRatings           0.0
author               0.0
bbeScore             0.0
bbeVotes             0.0
title                0.0
dtype: float64


In [9]:
from collections import Counter
import xml.etree.ElementTree as ET

print(open("data/dbpedia_book_data.xml", encoding="utf-8").read()[:1500])

root = ET.parse("data/dbpedia_book_data.xml").getroot()
local = lambda t: t.split("}")[-1]
print(local(root.tag), len(root))
print(Counter(local(e.tag) for e in root.iter()).most_common(15))

<?xml version="1.0" encoding="utf-8"?>
<books>
  <book>
    <title>Aurangzeb: The Man and the Myth</title>
    <publisher/>
    <year/>
    <pages>208</pages>
    <language/>
    <isbns>
      <isbn>978-0-14-344271-4</isbn>
    </isbns>
    <authors>
      <author>Audrey Truschke</author>
    </authors>
    <genres/>
  </book>
  <book>
    <title>A Fleeting Moment in My Country</title>
    <publisher/>
    <year/>
    <pages/>
    <language/>
    <isbns>
      <isbn>978-0984525546</isbn>
    </isbns>
    <authors/>
    <genres/>
  </book>
  <book>
    <title>Alias Grace</title>
    <publisher>Bloomsbury Publishing</publisher>
    <year/>
    <pages>470</pages>
    <language/>
    <isbns>
      <isbn>0-7710-0835-X</isbn>
    </isbns>
    <authors>
      <author>Margaret Atwood</author>
    </authors>
    <genres>
      <genre>Historical fiction</genre>
      <genre>Historical novel</genre>
    </genres>
  </book>
  <book>
    <title>American Pastoral</title>
    <publisher>Houghton Miff

In [5]:
chk = pd.read_csv("data/overlap_sample_check.csv")
print(chk.columns.tolist())
chk.head(20)

am[["title", "subtitle"]].dropna().head(5)      # Untertitel
gr["author"].sample(8, random_state=1)           # Autorenformate
am["authors"].sample(8, random_state=1)
for name, s in [("GR", gr["language"]), ("AM", am["language"]), ("DB", db.get("language"))]:
    if s is not None: print(name, s.dropna().value_counts().head(5))

['pair', 'key', 'title_left', 'title_right', 'true_match (y/n)']
GR language
English    42661
Arabic      1038
Spanish      687
French       579
German       528
Name: count, dtype: int64
AM language
English    32780
Spanish      591
German       288
French       200
Italian       88
Name: count, dtype: int64


In [8]:
from IPython.display import display
chk = pd.read_csv("data/overlap_sample_check.csv")
print(chk.columns.tolist())
display(chk.head(20))
display(am[["title", "subtitle"]].dropna().head(5))
print(gr["author"].sample(8, random_state=1).tolist())
print(am["authors"].sample(8, random_state=1).tolist())
for name, s in [("GR", gr["language"]), ("AM", am["language"]), ("DB", db.get("language"))]:
    if s is not None:
        print(name, s.dropna().value_counts().head(5).to_dict())

['pair', 'key', 'title_left', 'title_right', 'true_match (y/n)']


,pair,key,title_left,title_right,true_match (y/n)
0,Amazon_Goodreads,death on the nile|christie,Death on the Nile (Classic Reprint),Death on the Nile,y
1,Amazon_Goodreads,do no harm|marsh,"Do No Harm: Stories of Life, Death, and Brain ...","Do No Harm: Stories of Life, Death and Brain S...",y
2,Amazon_Goodreads,one small sacrifice|davidson,"One Small Sacrifice (Shadows of New York, 1)",One Small Sacrifice,y
3,Amazon_Goodreads,under different stars|bartol,Under Different Stars (The Kricket Series),Under Different Stars,y
4,Amazon_Goodreads,dead as a doornail|harris,Dead as a Doornail (Sookie Stackhouse/True Blo...,Dead as a Doornail,y
5,Amazon_Goodreads,one second after|forstchen,"One Second After (A John Matherson Novel, 1)",One Second After,y
6,Amazon_Goodreads,call of the wild|london,The Call of the Wild (Enriched Classics),The Call of the Wild,y
7,Amazon_Goodreads,bud not buddy|curtis,"Bud, Not Buddy","Bud, Not Buddy",y
8,Amazon_Goodreads,kindle fire owner s manual|weber,Kindle Fire Owner's Manual: The ultimate Kindl...,Kindle Fire Owner's Manual: The ultimate Kindl...,y
9,Amazon_Goodreads,hazel wood|albert,"The Hazel Wood (Hazel Wood, 1)",The Hazel Wood,y


,title,subtitle
0,Where the Crawdads Sing,"Paperback – Large Print, August 14, 2018"
1,The Girl on the Train,"Hardcover – Large Print, March 18, 2015"
2,Verity,"Hardcover – Special Edition, September 27, 2022"
3,It Ends with Us: Special Collector's Edition: ...,"Hardcover – Special Edition, April 18, 2023"
4,The Nightingale: A Novel,"Paperback – April 25, 2017"


['Carrie Pulkinen (Goodreads Author)', 'Krishan Chander, Кришан Чандар, В. Быкова (Translator), Е. Галей (Illustrator)', 'René Goscinny, Albert Uderzo (illustrator)', 'Terry Brooks (Goodreads Author)', 'Harold Klemp (Goodreads Author)', 'Will Durant, Ariel Durant, زكي نجيب محمود (Translator)', 'Saladin Ahmed (Goodreads Author), Javier Garrón (Illustrations)', 'Elmore Leonard']
['["Jodi Picoult"]', '["Elizabeth O\'Roark"]', '["Rick Riordan"]', '["Tessa Dare"]', '["James Patterson"]', '["Philip Kerr"]', '["Katie Fforde"]', '["Gaël Faye"]']
GR {'English': 42661, 'Arabic': 1038, 'Spanish': 687, 'French': 579, 'German': 528}
AM {'English': 32780, 'Spanish': 591, 'German': 288, 'French': 200, 'Italian': 88}


In [10]:
am["subtitle"].dropna().str.contains("Paperback|Hardcover|Kindle|Audio", case=False).mean()

np.float64(0.9294871794871795)

In [11]:
import xml.etree.ElementTree as ET
import pandas as pd, numpy as np

root = ET.parse("data/dbpedia_book_data.xml").getroot()
rows = []
for b in root.findall("book"):
    rows.append({
        "title": b.findtext("title"),
        "publisher": b.findtext("publisher"),
        "year": b.findtext("year"),
        "pages": b.findtext("pages"),
        "language": b.findtext("language"),
        "isbns": [i.text for i in b.findall("isbns/isbn")],
        "authors": [a.text for a in b.findall("authors/author")],
        "genres": [g.text for g in b.findall("genres/genre")],
    })
db = pd.DataFrame(rows).replace("", np.nan)
for c in ["isbns", "authors", "genres"]:
    db[c] = db[c].apply(lambda l: l if len(l) else np.nan)

print(len(db), len(db.columns))                       # 20981, 8
print((db.isna().mean() * 100).round(1).sort_values(ascending=False))
print(db["language"].dropna().value_counts().head(5).to_dict())
print(db["year"].dropna().head(5).tolist())

# Anteil Amazon-"subtitle" mit Format-/Datumsangaben
share = am["subtitle"].dropna().str.contains("Paperback|Hardcover|Kindle|Audio", case=False).mean()
print(round(share * 100, 1), "%")

20981 8
year         88.5
language     87.6
genres       49.1
publisher    25.4
pages        20.1
authors      13.5
title         0.0
isbns         0.0
dtype: float64
{'English language': 1458, 'French language': 79, 'German language': 60, 'Japanese language': 57, 'Spanish language': 56}
['1997', '2008', '1992', '2008', '1995']
92.9 %


In [5]:
import re
from pathlib import Path
from collections import Counter
import xml.etree.ElementTree as ET
import pandas as pd

BASE = Path.cwd()
summary, tops = [], {}

for name in ["goodreads.xml", "amazon.xml", "dbpedia.xml"]:
    total = with_br = series_like = has_series = 0
    contents = Counter()
    for _, el in ET.iterparse(BASE / "data" / name, events=("end",)):
        if el.tag != "book":
            continue
        total += 1
        title = el.findtext("title") or ""
        if (el.findtext("series") or "").strip():
            has_series += 1
        brackets = re.findall(r"\(([^()]*)\)", title)
        if brackets:
            with_br += 1
            for b in brackets:
                contents[re.sub(r"\d+", "#", b.strip())] += 1
            if any(re.search(r"(series|book|vol\.?|#)\s*,?\s*#?\d+", b, re.I) for b in brackets):
                series_like += 1
        el.clear()
    src = name.replace(".xml", "").capitalize()
    summary.append({
        "Source": src,
        "Books": total,
        "Series field filled": f"{has_series:,} ({has_series/total:.1%})",
        "Titles with brackets": f"{with_br:,} ({with_br/total:.1%})",
        "Numbered series in brackets": f"{series_like:,} ({series_like/total:.1%})",
    })
    tops[src] = [f"({b})  {n}" for b, n in contents.most_common(10)]

display(pd.DataFrame(summary).set_index("Source"))
display(pd.DataFrame(tops, index=range(1, 11)).rename_axis("Top bracket contents"))

,Books,Series field filled,Titles with brackets,Numbered series in brackets
Source,,,,
Goodreads,52478,"23,469 (44.7%)",948 (1.8%),348 (0.7%)
Amazon,40000,0 (0.0%),"15,356 (38.4%)","1,336 (3.3%)"
Dbpedia,59270,0 (0.0%),"12,595 (21.3%)",1 (0.0%)


,Goodreads,Amazon,Dbpedia
Top bracket contents,,,
1,(A Little Golden Book) 18,(#) 599,(novel) 6446
2,(#-#) 17,(Turtleback School & Library Binding Edition) ...,(book) 1857
3,(#) 9,(Spanish Edition) 177,(short story collection) 216
4,(Volume #) 9,(Random House Large Print) 140,(novella) 127
5,(Book #) 6,"(Alex Cross, #) 111",(poetry collection) 120
6,"(Psychological Wholeness Series, ##) 6",(In Death Series) 104,(anthology) 79
7,(German Edition) 6,(Book #) 102,(memoir) 78
8,(#-) 6,"(A Women's Murder Club Thriller, #) 102",(collection) 70
9,(Chicken Soup for the Soul) 5,"(A Harry Bosch Novel, #) 75",(short story) 51


In [7]:
import csv, re
from pathlib import Path
from collections import Counter
import xml.etree.ElementTree as ET
import pandas as pd
csv.field_size_limit(10**9)

BASE = Path.cwd()
SOURCES = {  # source: (file in data/, date column)
    "Goodreads": ("books_1.Best_Books_Ever.csv", "publishDate"),
    "DBpedia":   ("dbpedia_book_data.xml",       "release_date"),
    "Amazon":    ("amazon_books_subset.csv",     "publication_date"),
}
MON = r"(January|February|March|April|May|June|July|August|September|October|November|December)"
RULES = [  # order matters: first match wins
    ("Year only (completed to -01-01)", r"^\d{4}$"),
    ("Real 1 January", r"^01/01/(\d\d|\d{4})$|^January 1(st)?,? \d{4}$|^\d{4}-01-01$"),
    ("Other full date", r"^\d\d/\d\d/(\d\d|\d{4})$|^" + MON + r" \d{1,2}(st|nd|rd|th)?,? \d{4}$|^\d{4}-\d\d-\d\d$"),
]

def category(v):
    v = v.strip()
    if not v:
        return "Empty"
    for name, rx in RULES:
        if re.search(rx, v):
            return name
    return "Other (no full date)"

def values(path, col):
    if path.suffix == ".xml":
        for _, el in ET.iterparse(path, events=("end",)):
            if el.tag == "book":
                yield el.findtext(col) or ""
                el.clear()
    else:
        with open(path, newline="", encoding="utf-8") as f:
            for row in csv.DictReader(f):
                yield row.get(col) or ""

pd.set_option("display.max_colwidth", None)

for src, (file, col) in SOURCES.items():
    cnt, ex = Counter(), {}
    for v in values(BASE / "data" / file, col):
        k = category(v)
        cnt[k] += 1
        ex.setdefault(k, [])
        if len(ex[k]) < 3:
            ex[k].append(v.strip()[:30])
    total = sum(cnt.values())
    rows = [{"Category": k, "Count": n, "Share": f"{n/total:.1%}", "Examples": ex[k]}
            for k, n in cnt.most_common()]
    print(f"\n{src}: {total:,} books, column '{col}'")
    display(pd.DataFrame(rows).set_index("Category"))


Goodreads: 52,478 books, column 'publishDate'


,Count,Share,Examples
Category,,,
Other full date,42913,81.8%,"[09/14/08, 09/28/04, 05/23/06]"
Year only (completed to -01-01),4362,8.3%,"[2003, 1961, 1993]"
Other (no full date),3104,5.9%,"[December 2002, October 2003, July 2004]"
Real 1 January,1219,2.3%,"[01/01/04, 01/01/76, 01/01/10]"
Empty,880,1.7%,"[, , ]"



DBpedia: 59,270 books, column 'release_date'


,Count,Share,Examples
Category,,,
Year only (completed to -01-01),34355,58.0%,"[1982, 1985, 1986]"
Other full date,12880,21.7%,"[1982-04-15, 2009-05-18, 2001-06-04]"
Other (no full date),6250,10.5%,"[December 2005, --11-01, November 2008]"
Empty,5464,9.2%,"[, , ]"
Real 1 January,321,0.5%,"[2005-01-01, 2005-01-01, 2002-01-01]"



Amazon: 40,000 books, column 'publication_date'


,Count,Share,Examples
Category,,,
Other full date,31012,77.5%,"[April 30, 2019, August 14, 2018, June 28, 2022]"
Real 1 January,7022,17.6%,"[January 1, 2019, January 1, 2019, January 1, 2015]"
Empty,1966,4.9%,"[, , ]"


In [8]:
import csv, re
from pathlib import Path
from collections import Counter
import xml.etree.ElementTree as ET
import pandas as pd
csv.field_size_limit(10**9)

BASE = Path.cwd()
SOURCES = {
    "Goodreads": ("books_1.Best_Books_Ever.csv", "pages"),
    "Amazon":    ("amazon_books_subset.csv",     "pages"),
    "DBpedia":   ("dbpedia_book_data.xml",       "pages"),
}

def values(path, col):
    if path.suffix == ".xml":
        for _, el in ET.iterparse(path, events=("end",)):
            if el.tag == "book":
                yield el.findtext(col) or ""
                el.clear()
    else:
        with open(path, newline="", encoding="utf-8") as f:
            for row in csv.DictReader(f):
                yield row.get(col) or ""

pd.set_option("display.max_colwidth", None)
for src, (file, col) in SOURCES.items():
    cnt, ex = Counter(), {}
    for v in values(BASE / "data" / file, col):
        v = v.strip()
        k = "Empty" if not v else re.sub(r"\d+", "#", v)[:30]
        cnt[k] += 1
        ex.setdefault(k, [])
        if len(ex[k]) < 3:
            ex[k].append(v)
    total = sum(cnt.values())
    print(f"\n{src}: {total:,} books, column '{col}'")
    display(pd.DataFrame([{"Pattern": k, "Count": n, "Share": f"{n/total:.2%}", "Examples": ex[k]}
                          for k, n in cnt.most_common()]).set_index("Pattern"))


Goodreads: 52,478 books, column 'pages'


,Count,Share,Examples
Pattern,,,
#,50108,95.48%,"[374, 870, 324]"
Empty,2347,4.47%,"[, , ]"
# page,23,0.04%,"[1 page, 1 page, 1 page]"



Amazon: 40,000 books, column 'pages'


,Count,Share,Examples
Pattern,,,
#,28197,70.49%,"[496, 400, 400]"
Empty,11803,29.51%,"[, , ]"



DBpedia: 59,270 books, column 'pages'


,Count,Share,Examples
Pattern,,,
#,36517,61.61%,"[1176, 274, 229]"
Empty,22753,38.39%,"[, , ]"


In [9]:
import re
from pathlib import Path
from collections import Counter
import xml.etree.ElementTree as ET
import pandas as pd

BASE = Path.cwd()
rows, top = [], {}
for src in ["Goodreads", "Amazon", "DBpedia"]:
    total = has_series = with_brackets = 0
    contents = Counter()
    for _, el in ET.iterparse(BASE / "data" / f"{src.lower()}.xml", events=("end",)):
        if el.tag != "book":
            continue
        total += 1
        if (el.findtext("series") or "").strip():
            has_series += 1
        brackets = re.findall(r"\(([^()]*)\)", el.findtext("title") or "")
        if brackets:
            with_brackets += 1
            contents.update(re.sub(r"\d+", "#", b.strip()) for b in brackets)
        el.clear()
    rows.append({"Source": src, "Books": total,
                 "Series field": has_series, "Series field %": round(100 * has_series / total, 1),
                 "Titles with brackets": with_brackets,
                 "Titles with brackets %": round(100 * with_brackets / total, 1)})
    top[src] = [f"({b}) {n:,}" for b, n in contents.most_common(5)]

display(pd.DataFrame(rows).set_index("Source"))
display(pd.DataFrame(top, index=range(1, 6)).rename_axis("Most frequent bracket contents"))

,Books,Series field,Series field %,Titles with brackets,Titles with brackets %
Source,,,,,
Goodreads,52478,23469,44.7,948,1.8
Amazon,40000,0,0.0,15356,38.4
DBpedia,59270,0,0.0,12595,21.3


,Goodreads,Amazon,DBpedia
Most frequent bracket contents,,,
1,(A Little Golden Book) 18,(#) 599,"(novel) 6,446"
2,(#-#) 17,(Turtleback School & Library Binding Edition) 280,"(book) 1,857"
3,(#) 9,(Spanish Edition) 177,(short story collection) 216
4,(Volume #) 9,(Random House Large Print) 140,(novella) 127
5,(Book #) 6,"(Alex Cross, #) 111",(poetry collection) 120
